## 🔥 DAY 1 – Structured Output Enforcer

Build a LangChain system that:

✅ Enforces strict JSON output

✅ Uses a Pydantic schema

✅ Validates model responses

✅ Auto-retries on failure

✅ Includes 10 test prompts

In [1]:
from dateutil.rrule import parser
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.9
)

## How do we get reliable structured output from an LLM?

There are multiple approaches to generate a Structured output from an LLM.
Below are 4 such.

### Approach 1 - Manual String Prompting (f-string)

##### What This Approach Is

We manually write instructions inside the prompt.

We are trusting the model to:

- Read instructions
- Follow structure
- Not add extra text

In [2]:
topic = "Beyond Timescape"

manual_prompt = f"""
Return ONLY valid JSON. No markdown. No extra text.
Use exactly these keys: title, summary, confidence_score.
summary must be 2 sentences exactly and must be under 30 word count
confidence_score must be a number between 0 and 1.

Topic: {topic}
"""

In [3]:
response = llm.invoke(manual_prompt)
print(response.content)

{
  "title": "Beyond Timescape",
  "summary": "Exploring time travel theories and their implications. A new perspective on the fabric of space-time.",
  "confidence_score": 0.8
}


##### Observations

- Model returned valid JSON.
- No markdown wrapping.
- No extra text.
- Fields matched schema.
- confidence_score is valid float.

---

#### Result

- **Output:** Valid JSON
- **Extra text/markdown:** None
- **Schema match:** `title`, `summary`, `confidence_score`
- **Type correctness:** `confidence_score` is numeric
- **Takeaway:** Manual prompting can work, especially with a good model + clear instruction.

---

### Approach 2 — PromptTemplate

#### What we are doing?

Instead of manually building the prompt with an f-string

```
manual_prompt = f"... {topic} ..."
```

We use:

```</> Python
from langchain_core.prompts import PromptTemplate
```




In [4]:
from langchain_core.prompts import PromptTemplate

template = PromptTemplate.from_template("""
Return ONLY valid JSON. No markdown. No extra text.
Use exactly these keys: title, summary, confidence_score.
summary must be 2 sentences exactly and must be under 30 word count
confidence_score must be a number between 0 and 1.

Topic: {topic}
""")

formatted_prompt = template.format_prompt(topic="Beyond Timescape")
response = llm.invoke(formatted_prompt)
print(response.content)

{
  "title": "Beyond Timescape: A Multidimensional Exploration",
  "summary": "Beyond Timescape delves into uncharted territories of space and time. Exploring the unknown.",
  "confidence_score": 0.85
}


### Approach 3 — “Use the Pydantic schema itself in the prompt”

Instead of hand-writing the JSON structure in the prompt, we generate it automatically from your Pydantic class and show that to the model.

That way:

- The schema is the source of truth
- Your prompt stays consistent even if the class changes

It can be used alongside approach 1 or 2

In [5]:
from pydantic import BaseModel, Field

class StructuredResponse(BaseModel):
    title: str  = Field(description="Title of the topic")
    summary: str = Field(description="Clear explanation in 2 sentences exactly and must be under 30 word count")
    confidence_score: float = Field(description="Confidence score between 0 and 1")

In [6]:
import json

# Get Schema
schema_dict = StructuredResponse.model_json_schema()

# Convert schema dict -> JSON String
schema_text = json.dumps(schema_dict, indent=4)

topic = "Beyond Timescape"

In [7]:
# Approach 1 building prompt manually using schema as the contract
prompt = f"""
Return ONLY valid JSON that matches this JSON schema. No markdown. No extra text.
JSON Schema: {schema_text}
Topic: {topic}
"""

response = llm.invoke(prompt)
print(response.content)

{
    "title": "Beyond Timescape",
    "summary": "This article explores the concept of time travel and its implications on the fabric of reality. It also delves into the potential consequences of altering historical events.",
    "confidence_score": 0.8
}


In [8]:
# Approach 2 using the template
template = PromptTemplate.from_template("""
Return ONLY valid JSON that matches this JSON schema. No markdown. No extra text.
JSON Schema: {schema_text}
Topic: {topic}
""")

formatted_prompt = template.format(topic="Beyond Timescape", schema_text=schema_text)
response = llm.invoke(formatted_prompt)
print(response.content)

{
    "title": "Beyond Timescape",
    "summary": "This concept allows users to navigate through different time periods. It involves creating a virtual map of historical events.",
    "confidence_score": 0.85
}


### Approach 4 — PydanticOutputParser

##### What it does (in simple terms)

1. Takes your StructuredResponse model
2. Generates format instructions automatically from it
3. Forces the LLM to follow those instructions
4. Parses the output
5. Returns a real StructuredResponse object (not plain text)

In [9]:
from langchain_core.output_parsers import PydanticOutputParser

parser = PydanticOutputParser(pydantic_object=StructuredResponse)

prompt = PromptTemplate(
    template=(
        "Write a structured response about: {topic}\n\n"
        "{format_instructions}\n"
    ),
    input_variables=["topic"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

In [10]:
# Approach 1, step by step
prompt_value = prompt.format_prompt(topic="Beyond Timescape")
response = llm.invoke(prompt_value)
result = parser.invoke(response.content)
print(result)
print(type(result))

title="Beyond Time's Edge" summary="Beyond Time's Edge refers to a hypothetical concept that explores the nature of time and space. It delves into the realm of physics and theoretical constructs, questioning the fundamental laws governing our reality." confidence_score=0.8
<class '__main__.StructuredResponse'>


In [11]:
# Chain
chain = prompt | llm | parser
result = chain.invoke(prompt_value)
print(result)
print(type(result))

title='Beyond Timescape' summary='Beyond Timescape is a concept that redefines the boundaries of time and space. It proposes a multidimensional framework for understanding the fabric of reality.' confidence_score=0.85
<class '__main__.StructuredResponse'>


#### Observations:

- Uses Pydantic model as schema.
- Automatically generates formatting instructions.
- Parses model output into structured object.
- Validates fields and types.
- Throws error if invalid.
- More reliable than prompt-only enforcement.

---

##  ⚠️ But We Are Not Done Yet

Right now:

If the model fails → parser throws error.

In production, we don’t crash.

We:

1. Catch error
2. Send error back to model
3. Ask it to fix output
4. Retry
5. Only then fail

That is the true **Structured Output Enforcer**.

---
## Lets First intentionally break it and see how it fails

Lets use a “bad” prompt that encourages non-JSON

In [12]:
bad_prompt = PromptTemplate.from_template(
    """"Write a poetic paragraph about {topic}. Add bullets and a closing sentence.
    """)

bad_chain = bad_prompt | llm | parser

bad_chain.invoke(prompt_value)

OutputParserException: Invalid json output: In realms beyond Timescape, where data reigns,
A structured response is sought, with JSON's precise domains.
A schema is provided, a blueprint to abide,
To ensure the output is well-formatted, side by side.

• The schema dictates, with properties so fine,
A title, a summary, and a confidence score divine.
• The title, a string, with description so clear,
Must encapsulate the topic, without a speck of fear.
• The summary, a string, within the 30-word bound,
Must convey the essence, in two sentences, unbound.
• The confidence score, a number, so precise and true,
Must fall between 0 and 1, with no room for "I'm not sure".

This structured response, a JSON instance so grand,
Must conform to the schema, with no room for the hand.
It must be well-formatted, with properties in place,
For data to be trusted, and its secrets to be unveiled in space.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAIL

### Break Test — Parser Failure

- Bad prompt produced unstructured output (expected).
- Parser threw OutputParserException (expected).
- Takeaway: Parser acts as a gatekeeper. It rejects anything that doesn’t match the schema.

### What should we do now?

We must building a self-correcting system.

Instead of crashing, we:

1. Catch the parser error.
2. Show the model what it did wrong.
3. Ask it to fix only the format.
4. Retry.
5. Only fail if it still can’t fix itself even after multiple tries.

```
LLM → Bad Output → Parser Error
        ↓
     Repair Prompt
        ↓
LLM Fixes Output
        ↓
Parser Accepts
        ↓
Return Structured Object ✅
```

In [13]:
repair_prompt = PromptTemplate.from_template("""
You must output data that follows these formatting instructions:
{format_instructions}

The previous output did not follow the required format.

Error:
{error}

Bad output:
{bad_output}

Return ONLY the corrected output that matches the formatting instructions.
No markdown. No extra text.
""")

In [14]:
def generate_structured(topic: str, max_retries: int = 3):
    format_instructions = parser.get_format_instructions()
    last_error = None
    last_output = None

    for attempt in range(max_retries):
        # 1) try normal prompt on first attempt
        if attempt == 0:
            msg = llm.invoke(prompt.format_prompt(topic=topic))
            candidate = msg.content
        else:
            # 2) repair prompt on retries
            fix = repair_prompt.format_prompt(
                format_instructions=format_instructions,
                error=str(last_error),
                bad_output=last_output,
            )
            candidate = llm.invoke(fix).content

        # 3) parse/validate
        try:
            return parser.invoke(candidate)
        except Exception as e:
            last_error = e
            last_output = candidate

    # 4) fail gracefully
    raise RuntimeError(f"Failed after {max_retries} attempts. Last error: {last_error}")

In [25]:
result = generate_structured(topic="Beyond Timescape", max_retries=3)

In [26]:
print(
    f"Title: {result.title}\n\n"
    f"Summary:\n{result.summary}\n\n"
    f"Confidence Score: {result.confidence_score}"
)

Title: Beyond Timescape

Summary:
Beyond Timescape is a hypothetical concept that explores the idea of a multiverse, where every possible outcome of every event creates a new universe. This concept challenges our understanding of time and space, raising questions about the nature of reality and our place within it.

Confidence Score: 0.9


In [29]:
topics = [
    "LangChain",
    "RAG",
    "Neural Networks",
    "Climate Change",
    "Quantum Computing",
    "DevOps",
    "Microservices",
    "Blockchain",
    "Edge Computing",
    "Vector Databases"
]

for t in topics:
    print(f"\nTopic: {t}")
    result = generate_structured(t)
    print(
        f"Title: {result.title}\n"
        f"Summary:\n{result.summary}\n"
        f"Confidence Score: {result.confidence_score}\n"
    )


Topic: LangChain
Title: LangChain Overview
Summary:
LangChain is a Python library for building chain-like models, enabling users to create complex conversational interfaces. It allows users to integrate multiple models and chains of models in a flexible and scalable way.
Confidence Score: 0.8


Topic: RAG
Title: RAG Status
Summary:
A RAG status system used to indicate the level of completion or quality of a task or process, where green indicates completion or high quality, amber indicates partial completion or medium quality, and red indicates incomplete or low quality.
Confidence Score: 0.9


Topic: Neural Networks
Title: Neural Networks
Summary:
Neural networks are a type of machine learning algorithm inspired by the structure and function of the human brain. They consist of layers of interconnected nodes or 'neurons' that process and transmit information.
Confidence Score: 0.95


Topic: Climate Change
Title: Climate Change
Summary:
Climate change refers to the significant warming o